# 🎈 Balloon Detection — YOLOv8 & YOLOv11
## Hyperparameter optimization with Optuna + Experiment tracking with MLflow

**Pipeline:**
1. Setup: install dependencies, configure MLflow & Optuna
2. Define search space for hyperparameters
3. Optuna study: for each trial → train YOLO → log to MLflow
4. Compare experiments in MLflow UI
5. Load and evaluate the best model

---
**Dataset structure expected:**
```
dataset/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

## 0. Install dependencies

In [ ]:
!pip install ultralytics mlflow optuna optuna-integration[mlflow] plotly nbformat

## 1. Configuration

In [ ]:
import os
import mlflow
import optuna
from ultralytics import YOLO, settings
from pathlib import Path

# ─────────────────────────────────────────────
# 📁 PATHS — change to your actual paths
# ─────────────────────────────────────────────
DATA_YAML = "dataset/data.yaml"          # path to your data.yaml
RUNS_DIR  = "runs/balloon_detection"     # where YOLO saves weights

# ─────────────────────────────────────────────
# 📊 MLFLOW — tracking server
# Use local folder OR a remote server:
#   Local:   "runs/mlflow"
#   Remote:  "http://your-server-ip:5000"
# ─────────────────────────────────────────────
MLFLOW_TRACKING_URI   = "runs/mlflow"
MLFLOW_EXPERIMENT_NAME = "balloon_detection"

# ─────────────────────────────────────────────
# 🔬 STUDY CONFIG
# ─────────────────────────────────────────────
MODELS_TO_SEARCH = ["yolo11s.pt"]  # base checkpoints to try
N_TRIALS_PER_MODEL = 50   # Optuna trials per model (рекомендуется 30-50 для 12 параметров)
EPOCHS             = 30   # эпох на trial (короткие trials + pruning = быстрый поиск)
IMGSZ              = 640  # image size
METRIC_TO_OPTIMIZE = "metrics/mAP50(B)"  # metric logged by YOLO

# ─────────────────────────────────────────────
# ⚙️  ULTRALYTICS — enable MLflow integration
# ─────────────────────────────────────────────
settings.update({"mlflow": False})  # Отключаем автоинтеграцию — логируем вручную

# Configure MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"✅ MLflow tracking URI : {MLFLOW_TRACKING_URI}")
print(f"✅ Experiment name     : {MLFLOW_EXPERIMENT_NAME}")
print(f"✅ Models to search    : {MODELS_TO_SEARCH}")
print(f"✅ Trials per model    : {N_TRIALS_PER_MODEL}")

## 2. data.yaml helper (run only if you need to create/verify it)

In [ ]:
import yaml

# ── Verify data.yaml ──────────────────────────────────────────────────────────
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml contents:")
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

# Expected keys: train, val, test, nc (num classes), names
# Example data.yaml:
# train: dataset/train/images
# val:   dataset/val/images
# test:  dataset/test/images
# nc: 1
# names: [balloon]

## 3. Define hyperparameter search space

In [ ]:
def suggest_hyperparams(trial: optuna.Trial) -> dict:
    """
    Define the search space for Optuna.
    Adjust ranges based on your compute budget.
    """
    return {
        # Learning rate
        "lr0":       trial.suggest_float("lr0",    1e-5, 1e-1, log=True),
        "lrf":       trial.suggest_float("lrf",    0.01, 0.3),

        # Regularisation
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        "momentum":     trial.suggest_float("momentum",     0.8,  0.99),

        # Batch size
        "batch":     trial.suggest_categorical("batch", [16, 32]),

        # Optimizer
        "optimizer": trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW"]),

        # Augmentation
        "hsv_h":     trial.suggest_float("hsv_h",  0.0,  0.1),
        "hsv_s":     trial.suggest_float("hsv_s",  0.0,  0.9),
        "hsv_v":     trial.suggest_float("hsv_v",  0.0,  0.9),
        "flipud":    trial.suggest_float("flipud", 0.0,  0.5),
        "fliplr":    trial.suggest_float("fliplr", 0.0,  0.5),
        "mosaic":    trial.suggest_float("mosaic", 0.0,  1.0),
        "mixup":     trial.suggest_float("mixup",  0.0,  0.3),
    }

print("✅ Search space defined")

## 4. Objective function — one Optuna trial = one YOLO training run

### Межэпочный Pruning
После каждой эпохи колбэк `on_fit_epoch_end` передаёт текущий mAP50 в Optuna через `trial.report()`.
Если trial явно отстаёт от медианы уже завершённых trials — Optuna вызывает `TrialPruned` и немедленно
останавливает обучение. Это экономит 50–70% времени GPU.

In [ ]:
import threading

# ── Thread-local storage: каждый trial хранит свой объект trial изолированно ─
_thread_local = threading.local()


def make_objective(model_checkpoint: str):
    """
    Factory that returns an objective function bound to a specific YOLO checkpoint.
    Each call to objective() trains a new model and logs everything to MLflow.
    """
    def objective(trial: optuna.Trial) -> float:
        params = suggest_hyperparams(trial)

        run_name = (
            f"{Path(model_checkpoint).stem}"
            f"__trial{trial.number}"
            f"__lr{params['lr0']:.5f}"
            f"__bs{params['batch']}"
            f"__{params['optimizer']}"
        )

        # ── Страховка: закрыть любой незакрытый run от предыдущего trial ────
        try:
            mlflow.end_run()
        except Exception:
            pass

        # ── Сохранить trial в thread-local (для доступа из колбэка) ─────────
        _thread_local.trial   = trial
        _thread_local.pruned  = False

        with mlflow.start_run(run_name=run_name) as run:
            # ── Log all hyperparameters to MLflow ────────────────────────────
            mlflow.log_param("model",      model_checkpoint)
            mlflow.log_param("epochs",     EPOCHS)
            mlflow.log_param("imgsz",      IMGSZ)
            mlflow.log_param("trial",      trial.number)
            for k, v in params.items():
                mlflow.log_param(k, v)

            try:
                model = YOLO(model_checkpoint)

                # ── МЕЖЭПОЧНЫЙ PRUNING CALLBACK ──────────────────────────────
                # Вызывается Ultralytics после каждой эпохи.
                # Если Optuna решает что trial неперспективен → останавливаем.
                def on_fit_epoch_end(trainer):
                    epoch         = trainer.epoch  # 0-based
                    current_map50 = trainer.metrics.get("metrics/mAP50(B)", 0.0)

                    # Логируем в MLflow как временной ряд
                    mlflow.log_metric("epoch_mAP50", current_map50, step=epoch)

                    # Передаём результат в Optuna для pruning-решения
                    t = getattr(_thread_local, "trial", None)
                    if t is not None:
                        t.report(current_map50, step=epoch)
                        if t.should_prune():
                            _thread_local.pruned = True
                            raise optuna.TrialPruned()

                model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
                # ── END PRUNING CALLBACK ─────────────────────────────────────

                results = model.train(
                    data      = DATA_YAML,
                    epochs    = EPOCHS,
                    imgsz     = IMGSZ,
                    project   = RUNS_DIR,
                    name      = run_name,
                    exist_ok  = True,
                    verbose   = False,
                    **params
                )

                # ── Extract final metrics ────────────────────────────────────
                metrics = results.results_dict
                map50   = metrics.get(METRIC_TO_OPTIMIZE, 0.0)

                # ── Log final metrics to MLflow ──────────────────────────────
                mlflow.log_metric("mAP50",     metrics.get("metrics/mAP50(B)",    0.0))
                mlflow.log_metric("mAP50_95",  metrics.get("metrics/mAP50-95(B)", 0.0))
                mlflow.log_metric("precision", metrics.get("metrics/precision(B)", 0.0))
                mlflow.log_metric("recall",    metrics.get("metrics/recall(B)",   0.0))
                mlflow.log_metric("box_loss",  metrics.get("val/box_loss",        0.0))
                mlflow.log_metric("cls_loss",  metrics.get("val/cls_loss",        0.0))
                mlflow.log_metric("optuna_value", map50)
                mlflow.set_tag("status", "completed")

                # ── Save best weights as MLflow artifact ─────────────────────
                best_weights = Path(RUNS_DIR) / run_name / "weights" / "best.pt"
                if best_weights.exists():
                    mlflow.log_artifact(str(best_weights), artifact_path="weights")

                print(f"  ✅ Trial {trial.number} | {run_name} | mAP50={map50:.4f}")
                return map50

            except optuna.TrialPruned:
                mlflow.set_tag("status", "pruned")
                mlflow.set_tag("pruned_at_epoch", str(getattr(_thread_local, "trial", trial).number))
                print(f"  ✂️  Trial {trial.number} pruned")
                raise

            except Exception as e:
                mlflow.set_tag("status", "failed")
                mlflow.set_tag("error",  str(e))
                print(f"  ❌ Trial {trial.number} failed: {e}")
                return 0.0

    return objective


print("✅ Objective function с межэпочным pruning определена")


## 5. Run Optuna search for each model

In [ ]:
# Suppress verbose Optuna logs in notebook
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_studies = {}

for checkpoint in MODELS_TO_SEARCH:
    model_name = Path(checkpoint).stem
    study_name = f"{MLFLOW_EXPERIMENT_NAME}__{model_name}"

    print(f"\n{'='*60}")
    print(f"🔍 Starting search for: {checkpoint}")
    print(f"   Study name : {study_name}")
    print(f"   Trials     : {N_TRIALS_PER_MODEL}")
    print(f"{'='*60}")

    # MedianPruner cuts unpromising trials early → saves time
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=3,
        n_warmup_steps=10
    )

    study = optuna.create_study(
        study_name  = study_name,
        direction   = "maximize",
        pruner      = pruner,
        storage   = f"sqlite:///optuna_{model_name}.db",
        load_if_exists = True,
    )

    study.optimize(
        make_objective(checkpoint),
        n_trials   = N_TRIALS_PER_MODEL,
        show_progress_bar = True,
    )

    all_studies[model_name] = study

    print(f"\n🏆 Best trial for {model_name}:")
    print(f"   mAP50  : {study.best_value:.4f}")
    print(f"   Params : {study.best_params}")

print("\n✅ All studies complete!")

In [ ]:
import optuna
from pathlib import Path

all_studies = {}

for checkpoint in MODELS_TO_SEARCH:
    model_name = Path(checkpoint).stem
    study_name = f"{MLFLOW_EXPERIMENT_NAME}__{model_name}"
    db_path = f"sqlite:///optuna_{model_name}.db"

    study = optuna.load_study(
        study_name=study_name,
        storage=db_path,
    )
    all_studies[model_name] = study
    print(f"✅ Loaded: {model_name} — {len(study.trials)} trials, best mAP50={study.best_value:.4f}")


## 6. Results summary — compare all models

In [ ]:
import pandas as pd

rows = []
for model_name, study in all_studies.items():
    best = study.best_trial
    row  = {"model": model_name, "mAP50": best.value, "trial_number": best.number}
    row.update(best.params)
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("mAP50", ascending=False)
summary_df = summary_df.reset_index(drop=True)
summary_df.index += 1  # rank from 1

print("📊 Best hyperparameters per model (ranked by mAP50):")
print(summary_df.to_string())

In [ ]:
# ── Full trials dataframe for each model ────────────────────────────────────
for model_name, study in all_studies.items():
    df = study.trials_dataframe()
    print(f"\n📋 All trials for {model_name} (sorted by mAP50):")
    cols = ["number", "value", "state"] + [c for c in df.columns if c.startswith("params_")]
    available = [c for c in cols if c in df.columns]
    print(df[available].sort_values("value", ascending=False).to_string(index=False))

## 7. Optuna visualizations

In [ ]:
from optuna import visualization

for model_name, study in all_studies.items():
    print(f"\n📈 Optimization history — {model_name}")
    fig = visualization.plot_optimization_history(study)
    fig.update_layout(title=f"Optimization History — {model_name}")
    fig.show()

    print(f"\n🔑 Hyperparameter importance — {model_name}")
    fig = visualization.plot_param_importances(study)
    fig.update_layout(title=f"Hyperparameter Importance — {model_name}")
    fig.show()

    print(f"\n🌊 Parallel coordinate plot — {model_name}")
    fig = visualization.plot_parallel_coordinate(study)
    fig.update_layout(title=f"Parallel Coordinates — {model_name}")
    fig.show()

## 8. Query MLflow — load all experiment runs programmatically

In [ ]:
from mlflow.tracking import MlflowClient

client     = MlflowClient(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids = [experiment.experiment_id],
    order_by       = ["metrics.mAP50 DESC"],
)

# Keep only the most informative columns
metric_cols = [c for c in runs_df.columns if c.startswith("metrics.")]
param_cols  = [c for c in runs_df.columns if c.startswith("params.")] 
key_cols    = ["tags.mlflow.runName", "status"] + metric_cols + param_cols
available   = [c for c in key_cols if c in runs_df.columns]

print(f"Total runs found: {len(runs_df)}")
print("\n📊 Top 10 runs by mAP50:")
runs_df[available].head(10)

In [ ]:
# ── Overall best run ─────────────────────────────────────────────────────────
best_run = runs_df.iloc[0]

print("🏆 BEST RUN OVERALL")
print(f"   Run name  : {best_run.get('tags.mlflow.runName', best_run['run_id'])}")
print(f"   Model     : {best_run.get('params.model', 'N/A')}")
print(f"   mAP50     : {best_run.get('metrics.mAP50',    0):.4f}")
print(f"   mAP50-95  : {best_run.get('metrics.mAP50_95', 0):.4f}")
print(f"   Precision : {best_run.get('metrics.precision',0):.4f}")
print(f"   Recall    : {best_run.get('metrics.recall',   0):.4f}")
print(f"   lr0       : {best_run.get('params.lr0', 'N/A')}")
print(f"   batch     : {best_run.get('params.batch', 'N/A')}")
print(f"   optimizer : {best_run.get('params.optimizer', 'N/A')}")

BEST_RUN_ID = best_run["run_id"]

## 9. Download best weights from MLflow & run final evaluation

In [ ]:
import tempfile

# ── Download best.pt from MLflow artifact store ───────────────────────────────
artifacts = client.list_artifacts(BEST_RUN_ID, "weights")

if artifacts:
    local_dir = mlflow.artifacts.download_artifacts(
        run_id        = BEST_RUN_ID,
        artifact_path = "weights",
        dst_path      = "./best_model"
    )
    best_weights_path = str(Path(local_dir) / "best.pt")
    print(f"✅ Best weights downloaded to: {best_weights_path}")
else:
    # Fallback: find best.pt in YOLO runs directory
    run_name = best_run.get('tags.mlflow.runName', '')
    best_weights_path = str(Path(RUNS_DIR) / run_name / "weights" / "best.pt")
    print(best_weights_path)
    print(f"⚠️  Artifacts not found in MLflow, using local path: {best_weights_path}")

print(f"   Weights file exists: {Path(best_weights_path).exists()}")

In [ ]:
# ── Evaluate best model on test split ────────────────────────────────────────
best_model = YOLO(best_weights_path)

test_results = best_model.val(
    data   = DATA_YAML,
    split  = "test",
    imgsz  = IMGSZ,
    verbose= True,
)

# Log test metrics back to the best run
with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metric("test/mAP50",    test_results.results_dict.get("metrics/mAP50(B)",    0))
    mlflow.log_metric("test/mAP50_95", test_results.results_dict.get("metrics/mAP50-95(B)", 0))
    mlflow.log_metric("test/precision",test_results.results_dict.get("metrics/precision(B)",0))
    mlflow.log_metric("test/recall",   test_results.results_dict.get("metrics/recall(B)",   0))
    mlflow.set_tag("stage", "final_test")

print("\n🎯 Final test set metrics logged to MLflow run:", BEST_RUN_ID)

## 10. Run inference on sample images

In [ ]:
import glob
from IPython.display import Image as IPImage, display

# ── Pick a few images from the test set ──────────────────────────────────────
test_images = glob.glob("dataset/test/images/*.jpg")[:4]

if not test_images:
    print("⚠️  No test images found at dataset/test/images/*.jpg")
    print("   Change the path to point to your actual test images.")
else:
    results = best_model.predict(
        source     = test_images,
        imgsz      = IMGSZ,
        conf       = 0.25,
        save       = True,
        project    = "runs/predict",
        name       = "balloon_best_model",
        exist_ok   = True,
    )

    # Display predictions
    pred_images = glob.glob("runs/predict/balloon_best_model/*.jpg")
    for img_path in pred_images[:4]:
        display(IPImage(img_path, width=600))

## 11. Launch MLflow UI

Run the cell below to get the command to open MLflow UI in your browser.

> **Remote access**: if you're training on a server and want to view MLflow on your local machine, run:
> ```bash
> ssh -L 5000:localhost:5000 user@your-server-ip
> ```
> Then open `http://localhost:5000` in your browser.

In [ ]:
print("To open MLflow UI, run in terminal:")
print(f"\n  mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI} --port 5000\n")
print("Then open http://127.0.0.1:5000 in your browser.")
print("\nIn the UI you can:")
print("  • Select all runs → click 'Compare' to see a side-by-side table")
print("  • Use 'Chart' view to plot mAP50 vs lr0, batch, optimizer, etc.")
print("  • Download best.pt directly from the Artifacts tab of any run")

## 12. Export best model to ONNX (optional)

In [ ]:
# Uncomment to export the best model to ONNX format

# onnx_path = best_model.export(format="onnx", imgsz=IMGSZ)
# print(f"✅ Exported to ONNX: {onnx_path}")
#
# # Log ONNX file as MLflow artifact
# with mlflow.start_run(run_id=BEST_RUN_ID):
#     mlflow.log_artifact(onnx_path, artifact_path="export")
# print("✅ ONNX artifact logged to MLflow")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ── Лучшие параметры из Optuna ────────────────────────────────────────────────
best_model_name = max(all_studies, key=lambda m: all_studies[m].best_value)
best_trial      = all_studies[best_model_name].best_trial

# Убираем параметры которые зафиксируем вручную
final_params = {k: v for k, v in best_trial.params.items()
                if k not in ["flipud", "mosaic"]}

RUN_NAME = f"FINAL__{best_model_name}"

print(f"🏆 Модель   : {best_model_name}")
print(f"   Trial №  : {best_trial.number}")
print(f"   mAP50    : {best_trial.value:.4f}")
print(f"   Params   : {final_params}")

# ── Страховка: закрыть незакрытый MLflow run ──────────────────────────────────
try:
    mlflow.end_run()
except Exception:
    pass

# ── Финальное обучение ────────────────────────────────────────────────────────
final_model = YOLO(f"{best_model_name}.pt")

with mlflow.start_run(run_name=RUN_NAME) as run:

    # Логируем параметры
    mlflow.log_param("model",        best_model_name)
    mlflow.log_param("source_trial", best_trial.number)
    mlflow.log_param("source_mAP50", round(best_trial.value, 4))
    mlflow.log_param("epochs",       150)
    mlflow.log_param("patience",     30)
    mlflow.log_param("flipud",       0.0)
    mlflow.log_param("mosaic",       1.0)
    for k, v in final_params.items():
        mlflow.log_param(k, v)

    # ── Колбэк: логируем метрики по эпохам в MLflow ──────────────────────────
    def on_fit_epoch_end(trainer):
        epoch       = trainer.epoch
        map50       = trainer.metrics.get("metrics/mAP50(B)", 0.0)
        box_loss    = trainer.metrics.get("val/box_loss", 0.0)
        cls_loss    = trainer.metrics.get("val/cls_loss", 0.0)
        mlflow.log_metric("epoch_mAP50",   map50,    step=epoch)
        mlflow.log_metric("epoch_box_loss", box_loss, step=epoch)
        mlflow.log_metric("epoch_cls_loss", cls_loss, step=epoch)

    final_model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

    # ── Запуск обучения ───────────────────────────────────────────────────────
    results = final_model.train(
        data     = DATA_YAML,
        epochs   = 150,
        patience = 30,          # early stopping: стоп если 30 эпох нет улучшения
        imgsz    = IMGSZ,
        project  = RUNS_DIR,
        name     = RUN_NAME,
        exist_ok = True,
        verbose  = True,
        flipud   = 0.0,         # зафиксировано
        mosaic   = 1.0,         # зафиксировано
        **final_params
    )

    # ── Финальные метрики ─────────────────────────────────────────────────────
    metrics = results.results_dict
    mlflow.log_metric("final/mAP50",     metrics.get("metrics/mAP50(B)",    0))
    mlflow.log_metric("final/mAP50_95",  metrics.get("metrics/mAP50-95(B)", 0))
    mlflow.log_metric("final/precision", metrics.get("metrics/precision(B)", 0))
    mlflow.log_metric("final/recall",    metrics.get("metrics/recall(B)",   0))
    mlflow.log_metric("final/box_loss",  metrics.get("val/box_loss",        0))
    mlflow.log_metric("final/cls_loss",  metrics.get("val/cls_loss",        0))
    mlflow.set_tag("stage", "final_training")

    # ── Сохраняем веса (best.pt, и last.pt как запасной) ─────────────────────
    weights_dir  = Path(RUNS_DIR) / RUN_NAME / "weights"
    best_weights = weights_dir / "best.pt"
    last_weights = weights_dir / "last.pt"

    for pt_file in [best_weights, last_weights]:
        if pt_file.exists():
            mlflow.log_artifact(str(pt_file), artifact_path="weights")
            print(f"✅ Залогирован: {pt_file.name}")

    FINAL_RUN_ID  = run.info.run_id
    final_weights = best_weights if best_weights.exists() else last_weights

    print(f"\n🎯 Обучение завершено!")
    print(f"   Веса       : {final_weights}")
    print(f"   MLflow run : {FINAL_RUN_ID}")
    print(f"   mAP50      : {metrics.get('metrics/mAP50(B)', 0):.4f}")
    print(f"   mAP50-95   : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"   Precision  : {metrics.get('metrics/precision(B)', 0):.4f}")
    print(f"   Recall     : {metrics.get('metrics/recall(B)', 0):.4f}")